# Exploración: Realidad Aumentada con OpenCV y ArUco

Esta guía es un punto de partida para investigar realidad aumentada basada en marcadores usando Python. Está pensada para ser recorrida de manera personal antes de diseñar una actividad para el curso.

El stack que vamos a usar es:
- **OpenCV** (`cv2`): captura de video, detección de marcadores, transformaciones geométricas.
- **ArUco** (`cv2.aruco`): librería de marcadores para AR, incluida en OpenCV desde la versión 4.x.
- **NumPy**: operaciones matriciales sobre los frames y las transformaciones.

---
## Recursos de referencia

Antes de arrancar, estos son los recursos más sólidos para profundizar:

| Recurso | Descripción | Enlace |
|---|---|---|
| Documentación oficial ArUco | Referencia completa del módulo | https://docs.opencv.org/4.x/d5/dae/tutorial_aruco_detection.html |
| OpenCV-Python Tutorials | Tutoriales oficiales de OpenCV en Python | https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html |
| "Programming Computer Vision with Python" | Libro de Jan Erik Solem (libre) | http://programmingcomputervision.com/ |
| Repositorio `opencv-contrib-python` | Módulos extra incluyendo ArUco | https://pypi.org/project/opencv-contrib-python/ |

> **Nota:** Para usar ArUco se necesita `opencv-contrib-python`, no el paquete base `opencv-python`. Son excluyentes: si tenés el básico instalado, hay que desinstalarlo antes.

---
## 1. Instalación del entorno

Crear un entorno virtual nuevo para este proyecto es recomendable, ya que `opencv-contrib-python` puede entrar en conflicto con instalaciones previas de `opencv-python`.

In [ ]:
# Desinstalar opencv-python si está presente
!pip uninstall opencv-python -y

# Instalar la versión contrib que incluye ArUco
!pip install opencv-contrib-python numpy matplotlib

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

print(f"OpenCV versión: {cv2.__version__}")

# Verificar que el módulo aruco está disponible
try:
    aruco = cv2.aruco
    print("Módulo ArUco disponible.")
except AttributeError:
    print("ArUco no disponible. Revisar la instalación de opencv-contrib-python.")

---
## 2. ¿Qué es un diccionario ArUco?

Un **diccionario ArUco** es un conjunto predefinido de marcadores cuadrados binarios. Cada marcador tiene un ID único dentro del diccionario. La elección del diccionario determina cuántos marcadores distintos existen y qué tan fácil es distinguirlos entre sí.

Algunos diccionarios comunes:
- `DICT_4X4_50`: marcadores de 4×4 bits, 50 IDs disponibles. Simples y rápidos de detectar.
- `DICT_6X6_250`: marcadores de 6×6 bits, 250 IDs. Más robustos ante oclusiones parciales.
- `DICT_ARUCO_ORIGINAL`: el diccionario original de ArUco.

Para empezar, `DICT_4X4_50` es suficiente.

---
## 3. Generar e imprimir un marcador

El primer paso es generar el marcador para imprimirlo. El siguiente bloque crea el marcador con ID 0 del diccionario `DICT_4X4_50` y lo guarda como imagen PNG.

In [ ]:
# Seleccionar diccionario y parámetros del marcador
diccionario = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)

# Generar imagen del marcador (ID=0, tamaño=200x200 px, borde=1 celda)
marcador_img = cv2.aruco.generateImageMarker(diccionario, id=0, sidePixels=200, borderBits=1)

# Guardar el marcador como archivo
cv2.imwrite("marcador_aruco_id0.png", marcador_img)

# Visualizar en el cuaderno
plt.figure(figsize=(3, 3))
plt.imshow(marcador_img, cmap='gray')
plt.title("Marcador ArUco - ID 0")
plt.axis('off')
plt.show()

print("Marcador guardado como 'marcador_aruco_id0.png'. Imprimirlo en tamaño mínimo 5×5 cm.")

> **Para imprimir:** abrir el archivo `marcador_aruco_id0.png` y asegurarse de que el tamaño físico impreso sea de al menos 5×5 cm. Tamaños más grandes mejoran la detección a mayor distancia. Siempre conviene dejarlo sobre una superficie blanca con margen.

---
## 4. Detectar marcadores en una imagen estática

Antes de trabajar con video en tiempo real, conviene probar la detección sobre una imagen fija. Esto permite diagnosticar problemas de iluminación, tamaño o ángulo sin la complejidad del loop de video.

Para probarlo: fotografiá el marcador impreso con el celular y cargá la imagen en la misma carpeta que este cuaderno.

In [ ]:
# Cargar imagen con el marcador fotografiado
# Reemplazar 'foto_marcador.jpg' por el nombre real del archivo
imagen = cv2.imread('foto_marcador.jpg')

if imagen is None:
    print("No se encontró la imagen. Verificar el nombre y la ubicación del archivo.")
else:
    imagen_gris = cv2.cvtColor(imagen, cv2.COLOR_BGR2GRAY)

    diccionario = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
    parametros = cv2.aruco.DetectorParameters()
    detector = cv2.aruco.ArucoDetector(diccionario, parametros)

    # Detectar marcadores
    esquinas, ids, rechazados = detector.detectMarkers(imagen_gris)

    if ids is not None:
        print(f"Marcadores detectados: {ids.flatten()}")

        # Dibujar los marcadores detectados
        imagen_resultado = imagen.copy()
        cv2.aruco.drawDetectedMarkers(imagen_resultado, esquinas, ids)

        # Mostrar resultado
        imagen_rgb = cv2.cvtColor(imagen_resultado, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(8, 6))
        plt.imshow(imagen_rgb)
        plt.title(f"Marcadores detectados: {ids.flatten()}")
        plt.axis('off')
        plt.show()
    else:
        print("No se detectaron marcadores. Revisá iluminación, tamaño y ángulo del marcador.")

---
## 5. Detección en tiempo real con webcam

Este bloque abre la cámara y detecta marcadores frame por frame. Al correrlo, se abre una ventana independiente de OpenCV. Para cerrarla, presionar la tecla `q`.

> **Importante:** Este bloque **no funciona en Colab** (Colab no tiene acceso a la webcam del equipo). Hay que ejecutarlo como script local desde la terminal:
> ```bash
> python ar_webcam.py
> ```

In [ ]:
# ── Copiar este bloque en un archivo ar_webcam.py y ejecutarlo desde la terminal ──

import cv2
import numpy as np

diccionario = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
parametros = cv2.aruco.DetectorParameters()
detector = cv2.aruco.ArucoDetector(diccionario, parametros)

camara = cv2.VideoCapture(0)  # 0 = cámara por defecto

while True:
    ret, frame = camara.read()
    if not ret:
        print("No se pudo acceder a la cámara.")
        break

    gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    esquinas, ids, _ = detector.detectMarkers(gris)

    if ids is not None:
        cv2.aruco.drawDetectedMarkers(frame, esquinas, ids)

        # Por cada marcador detectado, mostrar su ID
        for i, esquina in enumerate(esquinas):
            c = esquina[0].mean(axis=0).astype(int)
            cv2.putText(frame, f"ID: {ids[i][0]}", tuple(c),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

    cv2.imshow("AR - Detección ArUco (q para salir)", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

camara.release()
cv2.destroyAllWindows()

---
## 6. Superposición de un objeto virtual (AR básico)

Detectar el marcador es solo el primer paso. El siguiente nivel es **superponer algo** sobre él: un rectángulo, texto, ejes de coordenadas o una figura geométrica que parezca estar "parada" sobre el marcador.

Para esto es necesaria la **calibración de la cámara**, que permite obtener los parámetros intrínsecos del lente (distancia focal, punto principal) necesarios para calcular la posición 3D del marcador relativa a la cámara.

El proceso general es:
1. Calibrar la cámara usando un tablero de ajedrez (OpenCV tiene utilidades para esto).
2. Usar `estimatePoseSingleMarkers()` para obtener los vectores de rotación y traslación del marcador en el espacio 3D.
3. Proyectar puntos 3D sobre el plano de la imagen con `projectPoints()` y dibujarlos.

Este es el punto de entrada a geometría proyectiva y visión por computadora, que es terreno fértil para profundizar en el curso.

### Recursos para continuar
- Calibración de cámara con OpenCV: https://docs.opencv.org/4.x/dc/dbb/tutorial_py_calibration.html
- Tutorial completo de AR con ArUco (incluye estimación de pose): https://pyimagesearch.com/2020/12/21/detecting-aruco-markers-with-opencv-and-python/
- Repositorio con ejemplos de AR en Python: https://github.com/niconielsen32/ComputerVision

---
## 7. Posibles actividades para el curso

Algunas ideas para estructurar este material como ejercicio de estudiantes, en orden creciente de complejidad:

1. **Generación e impresión del marcador:** introducción al concepto de marcador como dato visual codificado. Vínculo con la clase de representación de información en imágenes binarias.
2. **Detección en imagen estática:** los estudiantes fotografían el marcador y verifican la detección. Permite analizar factores que afectan la robustez (iluminación, ángulo, resolución).
3. **Detección en tiempo real:** trabajo con el loop de video, gestión de frames y el concepto de procesamiento en tiempo real.
4. **Superposición básica:** dibujar un cuadrado o texto sobre el marcador detectado. Introduce la proyección de coordenadas 2D en función de la posición del marcador.
5. **Calibración + estimación de pose:** proyecto integrador. Superponer un cubo 3D sobre el marcador, con perspectiva correcta.